# Transformers playground

A grab bag, not a lesson. Each part stands alone: run the setup, then jump to whatever you
need.

1. **The GPU.** Whether you have one, and what it changes.
2. **Hugging Face.** Where models come from, and the three lines that use one.
3. **Tokens.** What your text looks like to a model.
4. **ModernBERT.** Fill in a blank, then read a word's vector *in context*.
5. **Sentence embeddings.** Search 154 sonnets by meaning.
6. **Pictures and words together.** CLIP: search paintings by typing, and label them with no
   training.

Everything here runs on the free Colab tier. Models are downloaded the first time and kept for
the session.

> **Save a copy first:** File → Save a copy in Drive. Colab wipes its disk when the runtime
> ends.

In [ ]:
%pip install -q -U transformers sentence-transformers

In [ ]:
import os, time, re
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
print("torch", torch.__version__)

## 1 · The GPU

A GPU does thousands of multiplications at once. That is all a transformer does, so the same
code runs ten to fifty times faster on one.

In Colab: **Runtime → Change runtime type → T4 GPU**, then run this cell.

In [ ]:
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))
    print("memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
elif torch.backends.mps.is_available():          # Apple silicon, if you run this locally
    DEVICE = "mps"
    print("Apple GPU")
else:
    DEVICE = "cpu"
    print("No GPU. Everything below still works, just slower.")
    print("Colab: Runtime -> Change runtime type -> T4 GPU")
print("device:", DEVICE)

In [ ]:
# What the difference looks like: one big matrix multiply, on each device you have.
def time_matmul(device, n=2000, reps=5):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(reps):
        a @ b
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) / reps

cpu = time_matmul("cpu")
print(f"cpu   {cpu*1000:7.0f} ms")
if DEVICE != "cpu":
    gpu = time_matmul(DEVICE)
    print(f"{DEVICE:5s} {gpu*1000:7.0f} ms   ({cpu/gpu:.0f}x faster)")

## 2 · Hugging Face

[huggingface.co](https://huggingface.co/models) is where the models live. Each one has a name
like `answerdotai/ModernBERT-base`: the account, then the model. Click through to the **model
card** before you use anything — it says what the model was trained on, what it is for, and
what its licence allows.

The quickest way in is `pipeline`, which downloads the model, its tokenizer, and the code
around both.

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", device=DEVICE)
for line in ["This exhibition was a waste of an afternoon.",
             "I have not stopped thinking about the last room.",
             "It was, I suppose, a museum."]:
    out = sentiment(line)[0]
    print(f"{out['label']:8s} {out['score']:.2f}  {line}")

Two warnings about that cell.

It picked a model for you. For anything you will report, name the model yourself so you can
say which one you used. And a sentiment score is a guess from a model trained on product
reviews: read the third line above and decide whether you agree with it.

In [ ]:
# Where the downloads go. Delete this folder to reclaim the disk.
from pathlib import Path
cache = Path(os.environ.get("HF_HOME", Path.home() / ".cache/huggingface"))
print(cache)
if cache.exists():
    size = sum(f.stat().st_size for f in cache.rglob("*") if f.is_file())
    print(f"{size/1e9:.2f} GB cached so far")

## 3 · Tokens

A model never sees letters or words. It sees **tokens**: common words kept whole, rarer ones
split into pieces. That is why an unusual name costs more tokens than a common one, and why
models are quietly worse at words they have to spell out.

In [ ]:
from transformers import AutoTokenizer

MODEL = "answerdotai/ModernBERT-base"
tok = AutoTokenizer.from_pretrained(MODEL)

for line in ["The count invited us into his castle.",
             "Frankenstein; or, The Modern Prometheus",
             "unsupervised multimodal embeddings",
             "Whitby, Bistritz, Bukovina"]:
    pieces = [p.replace("\u0120", " ").strip() for p in tok.tokenize(line)]
    print(f"{len(pieces):3d} tokens |", " / ".join(pieces))
    print("            ", line, "\n")

In [ ]:
# Tokens are numbers. This is the actual input to every model in this notebook.
enc = tok("A museum keeps its collection.")
print(enc["input_ids"])
print(tok.convert_ids_to_tokens(enc["input_ids"]))

## 4 · ModernBERT

**ModernBERT** (2024) is a rebuilt BERT: same idea, better training, handles 8,192 tokens
instead of 512. It is an *encoder*. It does not write text. It reads a passage and hands back a
vector for every token, which is what you want for search, classification and clustering.

Its training task was filling in blanks, and you can still ask it to.

In [ ]:
fill = pipeline("fill-mask", model=MODEL, device=DEVICE)

for s in ["The count slept in a [MASK] during the day.",
          "Museums keep their collections in a [MASK].",
          "Mary Shelley wrote [MASK] in 1818.",
          "The scraper was blocked because it ignored [MASK]."]:
    guesses = [g["token_str"].strip() for g in fill(s, top_k=4)]
    print(f"{s:52s} {guesses}")

Look at the Mary Shelley line. It fills the blank with something grammatical, not something
true. An encoder learned which words fit where, not a library catalogue.

### The same word, in two contexts

Week 4's GloVe gave every word one vector for ever. A transformer gives it a different vector
in every sentence, and that is the whole difference.

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(MODEL).to(DEVICE).eval()

def word_vector(sentence, word):
    """The model's vector for one word, as used in this sentence."""
    enc = tok(sentence, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        states = model(**enc).last_hidden_state[0]           # one row per token
    pieces = tok.convert_ids_to_tokens(enc["input_ids"][0])
    hits = [i for i, p in enumerate(pieces)
            if p.replace("\u0120", "").lower().startswith(word[:4].lower())]
    v = states[hits].mean(0)
    return (v / v.norm()).cpu().numpy()

sentences = ["She sat on the grassy bank of the river.",
             "We moored the boat against the muddy bank.",
             "The bank refused to approve the loan.",
             "He works at the bank on the high street."]
V = np.stack([word_vector(s, "bank") for s in sentences])

print("     " + "".join(f"   s{i+1}" for i in range(len(sentences))))
for i, row in enumerate(V @ V.T):
    print(f"  s{i+1} " + "".join(f" {x:+.2f}" for x in row) + "   " + sentences[i])

The two river banks sit together, the two money banks sit together, and the two groups sit
apart. Same five letters, four different vectors.

## 5 · Sentence embeddings

One vector per token is a lot to carry around. A **sentence embedding** model squeezes a whole
passage into one vector, so you can compare passages directly.

`all-MiniLM-L6-v2` is small, fast and good enough for most coursework. Bigger ones are on the
[MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard).

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
demo = embedder.encode(["a walk by the river", "a stroll along the water",
                        "the interest rate went up"], normalize_embeddings=True)
print("each sentence is", demo.shape[1], "numbers")
print("river vs water:", f"{demo[0] @ demo[1]:+.2f}")
print("river vs rates:", f"{demo[0] @ demo[2]:+.2f}")

In [ ]:
# 154 sonnets, from the data folder in this repo.
def load_sonnets():
    for base in ("data/week01", "notebooks/data/week01", "../notebooks/data/week01"):
        path = os.path.join(base, "shakespeare_sonnets.txt")
        if os.path.exists(path):
            raw = open(path, encoding="utf-8", errors="ignore").read()
            break
    else:
        import requests
        raw = requests.get("https://www.gutenberg.org/cache/epub/1041/pg1041.txt",
                           timeout=30).text
    a, b = raw.find("*** START OF"), raw.find("*** END OF")
    body = raw[raw.find("\n", a) + 1:b]
    parts = re.split(r"\n\s*([IVXLC]+)\s*\n", body)          # roman-numeral headings
    out = []
    for i in range(1, len(parts) - 1, 2):
        text = " ".join(parts[i + 1].split())
        if len(text) > 200:
            out.append((parts[i], text))
    return out

sonnets = load_sonnets()
numbers = [n for n, _ in sonnets]
texts = [t for _, t in sonnets]
print(len(sonnets), "sonnets |", numbers[0], texts[0][:70], "...")

t0 = time.time()
S = embedder.encode(texts, normalize_embeddings=True, batch_size=32)
print(f"embedded in {time.time() - t0:.1f}s on {DEVICE}: {S.shape}")

In [ ]:
def search(question, k=3):
    scores = S @ embedder.encode([question], normalize_embeddings=True)[0]
    for i in np.argsort(-scores)[:k]:
        print(f"  {scores[i]:.2f}  {numbers[i]:>5}  {' '.join(texts[i].split()[:14])}...")

for q in ["time destroys beauty", "this poem will outlive us", "music", "jealousy"]:
    print("\n" + q)
    search(q)

None of those words have to appear in the poem. That is the difference between this and Week
2's counting.

### Your turn

In [ ]:
search("write your own question here", k=3)

In [ ]:
# The whole set, squashed to two dimensions.
from sklearn.decomposition import PCA

pts = PCA(n_components=2).fit_transform(S)
plt.figure(figsize=(8, 6))
plt.scatter(pts[:, 0], pts[:, 1], s=18, color="#A34526", alpha=0.75)
for i in range(0, len(pts), 12):
    plt.annotate(numbers[i], pts[i], xytext=(4, 3), textcoords="offset points", fontsize=9)
plt.xticks([]); plt.yticks([])
plt.title("154 sonnets, by meaning", loc="left")
plt.tight_layout(); plt.show()

## 6 · Pictures and words in one space

CLIP was trained on 400 million picture-caption pairs, pulling each picture towards its own
caption and away from everyone else's. The result is one space holding both, so a sentence and
an image get comparable vectors.

The 18 Met paintings below have no tags, no subjects, no descriptions. Only titles, which the
model never sees.

In [ ]:
import csv

def load_met():
    for base in ("data/week01", "notebooks/data/week01", "../notebooks/data/week01"):
        if os.path.isdir(os.path.join(base, "met")):
            rows = list(csv.DictReader(open(os.path.join(base, "met_manifest.csv"),
                                            encoding="utf-8")))
            return base, rows
    raise FileNotFoundError("run this from the repo, or clone it first")

base, rows = load_met()
paths = [os.path.join(base, r["file"]) for r in rows]
titles = [r["title"] for r in rows]

clip = SentenceTransformer("clip-ViT-B-32", device=DEVICE)
pictures = clip.encode([Image.open(p) for p in paths], normalize_embeddings=True)
print(len(paths), "paintings,", pictures.shape[1], "numbers each")

In [ ]:
def look_for(phrase, k=3):
    scores = pictures @ clip.encode([phrase], normalize_embeddings=True)[0]
    best = np.argsort(-scores)[:k]
    fig, axes = plt.subplots(1, k, figsize=(4 * k, 4))
    for ax, i in zip(axes, best):
        ax.imshow(Image.open(paths[i]))
        ax.set_title(f"{scores[i]:.2f}  {titles[i][:30]}", loc="left", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f'"{phrase}"', x=0.02, ha="left", fontsize=13)
    plt.tight_layout(); plt.show()

look_for("a man in a straw hat")
look_for("a woman in a white headdress")

### Labels with no training

Write the labels as sentences, embed them, and give each painting the nearest one. This is
**zero-shot** classification: no examples, no fitting, and you can change the label set by
editing a list.

In [ ]:
LABELS = ["a painting of a man", "a painting of a woman", "a landscape",
          "a religious painting", "a still life"]

label_vecs = clip.encode(LABELS, normalize_embeddings=True)
scores = pictures @ label_vecs.T
for i, t in enumerate(titles):
    j = int(np.argmax(scores[i]))
    print(f"{LABELS[j]:26s} {scores[i, j]:.2f}   {t[:46]}")

Look for Bronzino's *Portrait of a Young Man*. CLIP files it under woman, at 0.32, which is
the same score it gives the portraits it gets right. The score tells you how sure it is, not
whether it is correct.

Change `LABELS` and run it again. Try periods, moods, materials, whatever your project needs.

In [ ]:
# Which two paintings look most alike to CLIP? No text involved.
pairs = pictures @ pictures.T
np.fill_diagonal(pairs, -1)
i, j = np.unravel_index(np.argmax(pairs), pairs.shape)
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, k in zip(axes, (i, j)):
    ax.imshow(Image.open(paths[k]))
    ax.set_title(titles[k][:38], loc="left", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"closest pair: {pairs[i, j]:.2f}", x=0.02, ha="left", fontsize=13)
plt.tight_layout(); plt.show()

## Where to go next

- **Your own corpus.** Swap the sonnets for the CSV you collected in Week 4. Everything in
  Part 5 works on any list of strings.
- **A bigger embedder.** `all-mpnet-base-v2` is slower and better. The
  [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) ranks the rest.
- **Fine-tuning.** `cool-methods/finetune_modernbert.ipynb` trains ModernBERT on labels you
  made yourself. Do that only when a model somebody else trained has clearly failed you.
- **Other pipelines.** `pipeline("zero-shot-classification")`, `"ner"`, `"summarization"`,
  `"image-classification"`. Same three lines, different task name.

### Three things to keep saying out loud

1. Name the model and its version in anything you write. "An AI said" is not a method.
2. A model trained on the open web knows the open web. Neither Shakespeare nor your subreddit
   is the average of that.
3. A confident score is not a correct answer, and none of these models can tell you which of
   the two you are looking at.